# Metody analizy sieci złozonych

## Raport lista 1

---

### Zadanie nr 1 - zapoznanie się z podstawowymi narzędziami do analizy sieci złozonych oraz z podstawowymi miarami sieciowymi


### 1.1 Stworzenie pliku `asoiaf.csv` w oparciu o `out.asoiaf`

In [3]:
input_path = 'data/asoiaf/out.asoiaf'
output_path = 'data/asoiaf/asoiaf.csv'

with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
    for line in infile:
        if line.startswith('%'):
            continue

        parts = line.strip().split()

        if (len(parts) >= 2):
            outfile.write(f"{parts[0]},{parts[1]}\n")


### 1.2 Konfiguracja Neo4j

Wykonano następujące kroki w aplikacji Neo4j Desktop:

1. Utworzono nową lokalną bazę danych
2. Zainstalowano i włączono pluginy **APOC** i **Graph Data Science**
3. Plik `asoiaf.csv` przeniesiono do folderu `import` bazy danych

---

### 1.3 Import danych

Następnie wykonano ponizsze zapytania Cypher w aplikacji:

- zapewnienie unikalności węzłów:
```cypher
CREATE CONSTRAINT character_id IF NOT EXISTS
FOR (c:Character)
REQUIRE c.id IS UNIQUE;
```

- import węzłów i krawędzi z pliku CSV:
```cypher
LOAD CSV FROM 'file:///asoiaf.csv' AS row
WITH toInteger(row[0]) AS src, toInteger(row[1]) AS dst
MERGE (a:Character {id: src})
MERGE (b:Character {id: dst})
MERGE (a)-[:INTERACTS_WITH]->(b)
MERGE (b)-[:INTERACTS_WITH]->(a);
```

- stworzenie projekcji grafu:
```cypher
CALL gds.graph.project(
  'asoiafGraph',
  'Character',
  'INTERACTS_WITH'
)
```



### 1.4 Wizualizacja grafu

* Pełny graf:
![Pełny graf](results-ss/bloom-visualisation_0.png)

* Graf z Pagerank
![Graf z PageRank](results-ss/bloom-visualisation.png)

* Graf filtrowany (degree>12):
zastosowano równiez tutaj styl dla degree centrality
![Graf z degree>12](results-ss/bloom-visualisation_degree12.png)

### 1.5 Wierzchołki o najwyzszych wartościach zaaplikowanych centralności

Do pozyskania tych informacji wykorzystano zapytania Cypher:

* Pagerank

```cypher
CALL gds.pageRank.stream('asoiafGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).id AS node, score
ORDER BY score DESC
LIMIT 1
```

wynik:

| node | score              |
|------|--------------------|
| 60   | 14.500738434974942 |


* Degree centrality

```cypher
CALL gds.degree.stream('asoiafGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).id AS node, score
ORDER BY score DESC
LIMIT 1
```

wynik:

| node | score |
|------|-------|
| 11   | 120.0 |


Dyskusja po części 1:

- aplikacja wydaje się dosyć mało intuicyjna, wymaga trochę czasu na zapoznanie się i znalezienie potrzebych funkcji - szczególnie w zakładce Explore. Zakładkę Query natomiast oceniam całkiem intuicyjnie i czytelnie. Szczególny problem jak na pierwsze uzytkownie sprawiło odnalezienie niektórych funkcji związanych z analizą grafów w pluginie Graph Data Science, poniewaz nie wszystkie opcje są od razu widoczne lub wymagają wcześniejszego przygotowania projekcji grafu.
- uwazam, ze operacje wykonywaly sie szybko i płynnie - praktycznie od razu (wyjątek to zmiana layoutu na hierarchiczny - tu trzeba było chwilę poczekać)
- `force-based` layout uwidocznił skupiska/grupy węzłów oraz centralne punkty w sieci
- zmiana layoutu na `circular` pozwolila na latwiejsze zobrazowanie liczby polaczen miedzy node'ami w sposób bardziej uporządkowany, ale trudniej było w nim zauwazyć grupy w sieci:

![Graf z PageRank](results-ss/bloom-visualisation_circ.png)